# Notebook 05 — Binary Temporal Action Recognition

## SIH26174 Part 1

**Purpose:** train a focused binary temporal action-recognition model using pretrained **R(2+1)D-18**.

### Task

A 16-frame temporal clip is classified as:

- `0 = not_catch`
- `1 = catch`

Positive HMDB51 action: `catch`

Selected negative HMDB51 actions:

- `clap`
- `drink`
- `eat`
- `jump`
- `throw`

**Important limitation:** `not_catch` represents only those five selected negative HMDB51 actions. It does not represent every possible failed catch, laboratory error, or invalid experiment state.

The action model recognizes temporal visual content. Experiment sequence validation, `VALID/INVALID`, state-machine logic, alerts, voice output, YOLO, and pose/hand integration belong to later stages.

## Expected pipeline

```text
Notebook 02 authoritative video splits
                ↓
Filter selected HMDB51 actions
                ↓
Binary labels
                ↓
16-frame temporal clips
                ↓
R(2+1)D-18
                ↓
catch / not_catch
                ↓
Validation-based model selection
                ↓
Best checkpoint
```

Notebook 05 does **not** perform final test evaluation. The held-out test set is reserved for Notebook 06.

In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import gc
import json
import random
import time
import math

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision.models.video import (
    r2plus1d_18,
    R2Plus1D_18_Weights,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
)

print("Imports loaded.")

In [ ]:
# ============================================================
# 2. MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

print("Google Drive mounted.")

In [ ]:
# ============================================================
# 3. AUTHORITATIVE CONFIGURATION
# ============================================================

TRAINING_ROOT = Path(
    "/content/drive/MyDrive/SIH26174/training"
)

DATASET_ROOT = Path(
    "/content/drive/MyDrive/SIH26174/training/data/raw/hmdb51_sta"
)

SPLITS_DIR = TRAINING_ROOT / "data" / "splits"
CHECKPOINTS_DIR = TRAINING_ROOT / "checkpoints" / "action_model"
REPORTS_DIR = TRAINING_ROOT / "reports"
EXPORTS_DIR = TRAINING_ROOT / "exports"

for directory in [
    CHECKPOINTS_DIR,
    REPORTS_DIR,
    EXPORTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

POSITIVE_ACTION = "catch"

NEGATIVE_ACTIONS = [
    "clap",
    "drink",
    "eat",
    "jump",
    "throw",
]

TARGET_ACTIONS = [
    POSITIVE_ACTION,
    *NEGATIVE_ACTIONS,
]

CLASS_TO_IDX = {
    "not_catch": 0,
    "catch": 1,
}

IDX_TO_CLASS = {
    0: "not_catch",
    1: "catch",
}

NUM_CLASSES = 2

CLIP_LENGTH = 16
IMAGE_SIZE = 224

MEAN = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32,
)

STD = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32,
)

MODEL_NAME = "r2plus1d_18"
PRETRAINED = True
FREEZE_BACKBONE = True

BATCH_SIZE = 2
NUM_EPOCHS = 15

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

LR_MILESTONES = [8, 12]
LR_GAMMA = 0.1

GRAD_CLIP_MAX_NORM = 1.0

NUM_WORKERS = 2
PIN_MEMORY = True

USE_AMP = True
RANDOM_SEED = 42

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("NOTEBOOK 05 CONFIGURATION")
print("=" * 70)
print(f"Dataset root:      {DATASET_ROOT}")
print(f"Training root:     {TRAINING_ROOT}")
print(f"Model:             {MODEL_NAME}")
print(f"Pretrained:        {PRETRAINED}")
print(f"Freeze backbone:   {FREEZE_BACKBONE}")
print(f"Clip length:       {CLIP_LENGTH}")
print(f"Image size:        {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Batch size:        {BATCH_SIZE}")
print(f"Epochs:             {NUM_EPOCHS}")
print(f"Learning rate:     {LEARNING_RATE}")
print(f"Weight decay:      {WEIGHT_DECAY}")
print(f"AMP:               {USE_AMP}")
print(f"Device:            {DEVICE}")
print("=" * 70)

In [ ]:
# ============================================================
# 4. REPRODUCIBILITY + CONFIGURATION VALIDATION
# ============================================================

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(RANDOM_SEED)

assert MODEL_NAME == "r2plus1d_18"
assert NUM_CLASSES == 2
assert POSITIVE_ACTION == "catch"
assert CLASS_TO_IDX["not_catch"] == 0
assert CLASS_TO_IDX["catch"] == 1
assert CLIP_LENGTH == 16
assert IMAGE_SIZE == 224

assert DATASET_ROOT.exists()
assert (SPLITS_DIR / "train.csv").exists()
assert (SPLITS_DIR / "val.csv").exists()
assert (SPLITS_DIR / "test.csv").exists()

print("Configuration validation passed.")

In [ ]:
# ============================================================
# 5. LOAD AUTHORITATIVE NOTEBOOK 02 SPLITS
# ============================================================

train_df_raw = pd.read_csv(SPLITS_DIR / "train.csv")
val_df_raw = pd.read_csv(SPLITS_DIR / "val.csv")
test_df_raw = pd.read_csv(SPLITS_DIR / "test.csv")

print("Original Notebook 02 splits:")
print(f"  Train: {len(train_df_raw)}")
print(f"  Val:   {len(val_df_raw)}")
print(f"  Test:  {len(test_df_raw)}")

In [ ]:
# ============================================================
# 6. FILTER + BINARY MAPPING
# ============================================================

def filter_and_map_binary(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    required_columns = {"video_path", "class_name"}
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(
            f"Missing required columns: {sorted(missing)}"
        )

    df["original_action"] = (
        df["class_name"].astype(str).str.lower()
    )

    allowed = {
        action.lower()
        for action in TARGET_ACTIONS
    }

    df = df[
        df["original_action"].isin(allowed)
    ].copy()

    df["class_name"] = np.where(
        df["original_action"] == POSITIVE_ACTION,
        "catch",
        "not_catch",
    )

    df["class_id"] = df["class_name"].map(
        CLASS_TO_IDX
    )

    if df["class_id"].isna().any():
        raise ValueError(
            "Binary class mapping produced null class IDs."
        )

    df["class_id"] = df["class_id"].astype(int)

    return df.reset_index(drop=True)


train_df = filter_and_map_binary(train_df_raw)
val_df = filter_and_map_binary(val_df_raw)
test_df = filter_and_map_binary(test_df_raw)

print("Filtered binary datasets:")
for name, df in [
    ("Train", train_df),
    ("Val", val_df),
    ("Test", test_df),
]:
    print(f"\n{name}: {len(df)} videos")
    print(df["class_name"].value_counts().to_string())
    print("\nOriginal action distribution:")
    print(df["original_action"].value_counts().to_string())

In [ ]:
# ============================================================
# 7. DATA INTEGRITY / LEAKAGE CHECKS
# ============================================================

for name, df in [
    ("train", train_df),
    ("val", val_df),
    ("test", test_df),
]:
    duplicated = int(
        df["video_path"].duplicated().sum()
    )
    assert duplicated == 0, (
        f"{name} contains {duplicated} duplicate video paths."
    )

train_paths = set(train_df["video_path"])
val_paths = set(val_df["video_path"])
test_paths = set(test_df["video_path"])

assert train_paths.isdisjoint(val_paths)
assert train_paths.isdisjoint(test_paths)
assert val_paths.isdisjoint(test_paths)

for name, df in [
    ("train", train_df),
    ("val", val_df),
    ("test", test_df),
]:
    missing_paths = [
        p for p in df["video_path"]
        if not Path(p).exists()
    ]
    if missing_paths:
        raise FileNotFoundError(
            f"{name} contains {len(missing_paths)} missing video paths. "
            f"Example: {missing_paths[0]}"
        )

print("All duplicate-path and cross-split leakage checks passed.")

In [ ]:
# ============================================================
# 8. ACTUAL TRAINING CLASS WEIGHTS
# ============================================================

class_counts = (
    train_df["class_id"]
    .value_counts()
    .sort_index()
)

count_not_catch = int(
    class_counts.get(0, 0)
)

count_catch = int(
    class_counts.get(1, 0)
)

assert count_not_catch > 0
assert count_catch > 0

total_train = count_not_catch + count_catch

class_weights = torch.tensor(
    [
        total_train / (2.0 * count_not_catch),
        total_train / (2.0 * count_catch),
    ],
    dtype=torch.float32,
    device=DEVICE,
)

print("Training class counts:")
print(f"  not_catch: {count_not_catch}")
print(f"  catch:     {count_catch}")

print("\nComputed CrossEntropy class weights:")
print(f"  not_catch: {class_weights[0].item():.4f}")
print(f"  catch:     {class_weights[1].item():.4f}")

## Dataset design

The dataset does not decode and retain every frame of a video in RAM.

Training:
- random contiguous 16-frame temporal crop when possible

Validation:
- deterministic uniform sampling across the complete video

For short videos, temporal indices are repeated so exactly 16 frames are returned.

All outputs are:

```text
[C,T,H,W] = [3,16,224,224]
```

In [ ]:
# ============================================================
# 9. TEMPORAL DATASET
# ============================================================

class BinaryTemporalActionDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        clip_length: int = CLIP_LENGTH,
        image_size: int = IMAGE_SIZE,
        mean: np.ndarray = MEAN,
        std: np.ndarray = STD,
        is_train: bool = True,
        horizontal_flip_prob: float = 0.5,
        brightness_range: float = 0.05,
        contrast_range: float = 0.05,
    ):
        self.df = df.reset_index(drop=True)
        self.clip_length = clip_length
        self.image_size = image_size
        self.mean = mean.reshape(1, 1, 3)
        self.std = std.reshape(1, 1, 3)
        self.is_train = is_train

        self.horizontal_flip_prob = horizontal_flip_prob
        self.brightness_range = brightness_range
        self.contrast_range = contrast_range

    def __len__(self):
        return len(self.df)

    @staticmethod
    def _get_frame_count(cap):
        return max(
            int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
            0,
        )

    def _choose_indices(self, total_frames: int):
        if total_frames <= 0:
            return None

        if total_frames < self.clip_length:
            return np.linspace(
                0,
                total_frames - 1,
                self.clip_length,
            ).round().astype(int)

        if self.is_train:
            max_start = (
                total_frames - self.clip_length
            )
            start = random.randint(
                0,
                max_start,
            )

            return np.arange(
                start,
                start + self.clip_length,
                dtype=int,
            )

        return np.linspace(
            0,
            total_frames - 1,
            self.clip_length,
        ).round().astype(int)

    def _read_indices(
        self,
        video_path: Path,
        indices: np.ndarray,
    ):
        cap = cv2.VideoCapture(
            str(video_path)
        )

        if not cap.isOpened():
            raise RuntimeError(
                f"Could not open video: {video_path}"
            )

        frames = []

        try:
            contiguous = (
                len(indices) > 1
                and np.all(
                    np.diff(indices) == 1
                )
            )

            if self.is_train and contiguous:
                cap.set(
                    cv2.CAP_PROP_POS_FRAMES,
                    int(indices[0]),
                )

                last_frame = None

                for _ in indices:
                    ok, frame = cap.read()

                    if ok:
                        frame = cv2.cvtColor(
                            frame,
                            cv2.COLOR_BGR2RGB,
                        )
                        last_frame = frame
                    elif last_frame is not None:
                        frame = last_frame.copy()
                    else:
                        raise RuntimeError(
                            f"Could not decode frames from: "
                            f"{video_path}"
                        )

                    frames.append(frame)

            else:
                last_frame = None

                for index in indices:
                    cap.set(
                        cv2.CAP_PROP_POS_FRAMES,
                        int(index),
                    )

                    ok, frame = cap.read()

                    if ok:
                        frame = cv2.cvtColor(
                            frame,
                            cv2.COLOR_BGR2RGB,
                        )
                        last_frame = frame
                    elif last_frame is not None:
                        frame = last_frame.copy()
                    else:
                        raise RuntimeError(
                            f"Could not decode frame {index} "
                            f"from {video_path}"
                        )

                    frames.append(frame)

        finally:
            cap.release()

        if len(frames) != self.clip_length:
            raise RuntimeError(
                f"Expected {self.clip_length} frames, "
                f"got {len(frames)} for {video_path}"
            )

        return frames

    def _apply_clip_augmentation(
        self,
        clip: np.ndarray,
    ):
        clip = clip.copy()

        if not self.is_train:
            return clip

        if (
            random.random()
            < self.horizontal_flip_prob
        ):
            clip = (
                clip[:, :, ::-1, :]
                .copy()
            )

        brightness = random.uniform(
            -self.brightness_range,
            self.brightness_range,
        )

        contrast = random.uniform(
            1.0 - self.contrast_range,
            1.0 + self.contrast_range,
        )

        clip_float = (
            clip.astype(np.float32)
            / 255.0
        )

        clip_float = (
            clip_float * contrast
            + brightness
        )

        clip_float = np.clip(
            clip_float,
            0.0,
            1.0,
        )

        return (
            clip_float * 255.0
        ).round().astype(np.uint8)

    def _preprocess_clip(
        self,
        frames,
    ):
        processed = []

        for frame in frames:
            frame = cv2.resize(
                frame,
                (
                    self.image_size,
                    self.image_size,
                ),
                interpolation=cv2.INTER_LINEAR,
            )

            frame = (
                frame.astype(np.float32)
                / 255.0
            )

            frame = (
                frame - self.mean
            ) / self.std

            processed.append(frame)

        clip = np.stack(
            processed,
            axis=0,
        )

        # [T,H,W,C] -> [C,T,H,W]
        clip = np.transpose(
            clip,
            (3, 0, 1, 2),
        )

        return torch.from_numpy(
            clip.copy()
        ).float()

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        video_path = Path(
            row["video_path"]
        )

        class_name = row["class_name"]
        class_id = int(
            row["class_id"]
        )

        cap = cv2.VideoCapture(
            str(video_path)
        )

        if not cap.isOpened():
            raise RuntimeError(
                f"Could not open video: {video_path}"
            )

        try:
            total_frames = self._get_frame_count(
                cap
            )
        finally:
            cap.release()

        indices = self._choose_indices(
            total_frames
        )

        if indices is None:
            raise RuntimeError(
                f"No readable frames found for: "
                f"{video_path}"
            )

        frames = self._read_indices(
            video_path,
            indices,
        )

        clip = np.stack(
            frames,
            axis=0,
        )

        clip = self._apply_clip_augmentation(
            clip
        )

        clip_tensor = self._preprocess_clip(
            clip
        )

        return {
            "clip": clip_tensor,
            "class_id": torch.tensor(
                class_id,
                dtype=torch.long,
            ),
            "class_name": class_name,
            "original_action": row[
                "original_action"
            ],
            "video_path": str(video_path),
        }

In [ ]:
# ============================================================
# 10. DATASET SANITY CHECK
# ============================================================

train_dataset = BinaryTemporalActionDataset(
    train_df,
    is_train=True,
)

val_dataset = BinaryTemporalActionDataset(
    val_df,
    is_train=False,
)

sample = train_dataset[0]

expected_shape = (
    3,
    CLIP_LENGTH,
    IMAGE_SIZE,
    IMAGE_SIZE,
)

assert sample["clip"].shape == expected_shape
assert sample["class_id"].item() in [0, 1]

print("Single-sample sanity check passed.")
print(f"  Clip shape: {tuple(sample['clip'].shape)}")
print(f"  Label:      {sample['class_name']}")
print(f"  Original:   {sample['original_action']}")

In [ ]:
# ============================================================
# 11. DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY and DEVICE.type == "cuda",
    drop_last=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY and DEVICE.type == "cuda",
    drop_last=False,
)

batch = next(iter(train_loader))

assert batch["clip"].ndim == 5
assert batch["clip"].shape[1:] == (
    3,
    CLIP_LENGTH,
    IMAGE_SIZE,
    IMAGE_SIZE,
)

print("DataLoader sanity check passed.")
print(f"  Train dataset: {len(train_dataset)}")
print(f"  Val dataset:   {len(val_dataset)}")
print(f"  Batch shape:   {tuple(batch['clip'].shape)}")
print(f"  Labels:        {batch['class_id'].tolist()}")

## Model

The model is **only** torchvision R(2+1)D-18.

The pretrained backbone is frozen for the initial transfer-learning experiment; only the final binary classifier head is trained.

In [ ]:
# ============================================================
# 12. R(2+1)D-18 MODEL
# ============================================================

weights = (
    R2Plus1D_18_Weights.DEFAULT
    if PRETRAINED
    else None
)

model = r2plus1d_18(
    weights=weights
)

in_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(
        in_features,
        NUM_CLASSES,
    ),
)

if FREEZE_BACKBONE:
    for parameter in model.parameters():
        parameter.requires_grad = False

    for parameter in model.fc.parameters():
        parameter.requires_grad = True

model = model.to(DEVICE)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

frozen_params = (
    total_params - trainable_params
)

assert trainable_params > 0

print("=" * 70)
print("R(2+1)D-18 MODEL")
print("=" * 70)
print(f"Total parameters:      {total_params:,}")
print(f"Trainable parameters:  {trainable_params:,}")
print(f"Frozen parameters:     {frozen_params:,}")
print(f"Device:                {DEVICE}")
print("=" * 70)

In [ ]:
# ============================================================
# 13. LOSS / OPTIMIZER / SCHEDULER / AMP
# ============================================================

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

trainable_parameters = [
    p
    for p in model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=LR_MILESTONES,
    gamma=LR_GAMMA,
)

amp_enabled = (
    USE_AMP
    and DEVICE.type == "cuda"
)

if amp_enabled:
    try:
        scaler = torch.amp.GradScaler("cuda")
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler()
else:
    scaler = None

print("Training components initialized.")
print("  Loss:          weighted CrossEntropyLoss")
print("  Optimizer:     AdamW")
print("  Scheduler:     MultiStepLR")
print(f"  AMP enabled:   {amp_enabled}")

## Tiny-set pipeline sanity check

This checks that the training pipeline is mechanically capable of reducing loss on a deliberately tiny two-class subset.

It is **not** a model-performance result.

In [ ]:
# ============================================================
# 14. TINY-SET OVERFIT SANITY CHECK
# ============================================================

RUN_TINY_SET_SANITY_CHECK = True

if RUN_TINY_SET_SANITY_CHECK:
    tiny_n_per_class = min(
        4,
        int((train_df["class_id"] == 0).sum()),
        int((train_df["class_id"] == 1).sum()),
    )

    assert tiny_n_per_class >= 1

    tiny_df = pd.concat(
        [
            train_df[
                train_df["class_id"] == 0
            ].head(tiny_n_per_class),
            train_df[
                train_df["class_id"] == 1
            ].head(tiny_n_per_class),
        ],
        ignore_index=True,
    )

    tiny_dataset = BinaryTemporalActionDataset(
        tiny_df,
        is_train=False,
    )

    tiny_loader = DataLoader(
        tiny_dataset,
        batch_size=min(2, len(tiny_dataset)),
        shuffle=True,
        num_workers=0,
    )

    tiny_model = r2plus1d_18(
        weights=None
    )

    tiny_in_features = (
        tiny_model.fc.in_features
    )

    tiny_model.fc = nn.Linear(
        tiny_in_features,
        NUM_CLASSES,
    )

    tiny_model = tiny_model.to(DEVICE)

    tiny_optimizer = torch.optim.AdamW(
        tiny_model.parameters(),
        lr=1e-3,
        weight_decay=0.0,
    )

    tiny_criterion = nn.CrossEntropyLoss()

    tiny_model.train()

    initial_loss = None
    final_loss = None

    for step in range(25):
        running_loss = 0.0
        count = 0

        for tiny_batch in tiny_loader:
            clips = tiny_batch["clip"].to(
                DEVICE,
                non_blocking=True,
            )

            labels = tiny_batch["class_id"].to(
                DEVICE,
                non_blocking=True,
            )

            tiny_optimizer.zero_grad(
                set_to_none=True
            )

            logits = tiny_model(
                clips
            )

            loss = tiny_criterion(
                logits,
                labels,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                tiny_model.parameters(),
                1.0,
            )

            tiny_optimizer.step()

            running_loss += (
                loss.item()
                * labels.size(0)
            )

            count += labels.size(0)

        epoch_loss = (
            running_loss / count
        )

        if initial_loss is None:
            initial_loss = epoch_loss

        final_loss = epoch_loss

    print("Tiny-set sanity check finished.")
    print(
        f"  Initial loss: {initial_loss:.4f}"
    )
    print(
        f"  Final loss:   {final_loss:.4f}"
    )

    if final_loss >= initial_loss:
        raise RuntimeError(
            "Tiny-set sanity check failed: "
            "loss did not decrease."
        )

    print(
        "Tiny-set pipeline sanity check passed."
    )

    del tiny_model
    del tiny_optimizer
    del tiny_loader
    del tiny_dataset

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("Tiny-set sanity check skipped.")

In [ ]:
# ============================================================
# 15. TRAINING / VALIDATION FUNCTIONS
# ============================================================

def prepare_batch(batch):
    clips = batch["clip"].to(
        DEVICE,
        non_blocking=True,
    )

    labels = batch["class_id"].to(
        DEVICE,
        non_blocking=True,
    )

    if clips.ndim != 5:
        raise ValueError(
            f"Expected 5D clips, got {tuple(clips.shape)}"
        )

    if clips.shape[1] != 3:
        raise ValueError(
            "Expected [B,C,T,H,W] input."
        )

    return clips, labels


def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch in loader:
        clips, labels = prepare_batch(
            batch
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        if scaler is not None:
            with torch.amp.autocast(
                device_type="cuda"
            ):
                logits = model(clips)
                loss = criterion(
                    logits,
                    labels,
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP_MAX_NORM,
            )

            scaler.step(optimizer)
            scaler.update()

        else:
            logits = model(clips)

            loss = criterion(
                logits,
                labels,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP_MAX_NORM,
            )

            optimizer.step()

        batch_size = labels.size(0)

        running_loss += (
            loss.item()
            * batch_size
        )

        predictions = logits.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += batch_size

    return (
        running_loss / total,
        correct / total,
    )


@torch.no_grad()
def validate_one_epoch(
    model,
    loader,
    criterion,
):
    model.eval()

    running_loss = 0.0
    total = 0

    all_preds = []
    all_labels = []
    all_catch_probs = []

    for batch in loader:
        clips, labels = prepare_batch(
            batch
        )

        if amp_enabled:
            with torch.amp.autocast(
                device_type="cuda"
            ):
                logits = model(clips)
                loss = criterion(
                    logits,
                    labels,
                )
        else:
            logits = model(clips)
            loss = criterion(
                logits,
                labels,
            )

        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        predictions = probabilities.argmax(
            dim=1
        )

        batch_size = labels.size(0)

        running_loss += (
            loss.item()
            * batch_size
        )

        total += batch_size

        all_preds.extend(
            predictions.cpu().numpy().tolist()
        )

        all_labels.extend(
            labels.cpu().numpy().tolist()
        )

        all_catch_probs.extend(
            probabilities[:, 1]
            .cpu()
            .numpy()
            .tolist()
        )

    return (
        running_loss / total,
        np.asarray(all_preds),
        np.asarray(all_labels),
        np.asarray(all_catch_probs),
    )


def calculate_metrics(
    labels,
    predictions,
):
    return {
        "accuracy": float(
            accuracy_score(
                labels,
                predictions,
            )
        ),
        "precision": float(
            precision_score(
                labels,
                predictions,
                pos_label=1,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                labels,
                predictions,
                pos_label=1,
                zero_division=0,
            )
        ),
        "f1": float(
            f1_score(
                labels,
                predictions,
                pos_label=1,
                zero_division=0,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                labels,
                predictions,
            )
        ),
    }


print("Training and validation functions defined.")

In [ ]:
# ============================================================
# 16. CHECKPOINT FUNCTION
# ============================================================

def save_checkpoint(
    epoch,
    model,
    optimizer,
    scheduler,
    scaler,
    val_loss,
    metrics,
    is_best,
):
    checkpoint = {
        "epoch": int(epoch),

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "scaler_state_dict":
            (
                scaler.state_dict()
                if scaler is not None
                else None
            ),

        "val_loss": float(val_loss),
        "val_accuracy": float(
            metrics["accuracy"]
        ),
        "val_precision": float(
            metrics["precision"]
        ),
        "val_recall": float(
            metrics["recall"]
        ),
        "val_f1": float(
            metrics["f1"]
        ),
        "val_balanced_accuracy": float(
            metrics["balanced_accuracy"]
        ),

        "model_name": MODEL_NAME,
        "pretrained": PRETRAINED,
        "freeze_backbone": FREEZE_BACKBONE,

        "class_to_idx": CLASS_TO_IDX,
        "idx_to_class": IDX_TO_CLASS,

        "positive_action": POSITIVE_ACTION,
        "negative_actions": NEGATIVE_ACTIONS,
        "target_actions": TARGET_ACTIONS,

        "clip_length": CLIP_LENGTH,
        "image_size": IMAGE_SIZE,
        "mean": MEAN.tolist(),
        "std": STD.tolist(),

        "random_seed": RANDOM_SEED,
    }

    last_path = (
        CHECKPOINTS_DIR
        / "last_checkpoint.pt"
    )

    torch.save(
        checkpoint,
        last_path,
    )

    best_path = None

    if is_best:
        best_path = (
            CHECKPOINTS_DIR
            / "best_checkpoint.pt"
        )

        torch.save(
            checkpoint,
            best_path,
        )

    return last_path, best_path

## Full training

The best model is selected using **validation catch F1**.

The test split is never decoded or used for model selection in this notebook.

In [ ]:
# ============================================================
# 17. MAIN TRAINING LOOP
# ============================================================

history = {
    "epoch": [],
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": [],
    "val_balanced_accuracy": [],
    "learning_rate": [],
    "epoch_seconds": [],
}

best_val_f1 = -float("inf")
best_epoch = 0

training_start = time.time()

for epoch in range(
    1,
    NUM_EPOCHS + 1,
):
    epoch_start = time.time()

    train_loss, train_accuracy = (
        train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
        )
    )

    (
        val_loss,
        val_preds,
        val_labels,
        val_catch_probs,
    ) = validate_one_epoch(
        model=model,
        loader=val_loader,
        criterion=criterion,
    )

    metrics = calculate_metrics(
        val_labels,
        val_preds,
    )

    is_best = (
        metrics["f1"] > best_val_f1
    )

    if is_best:
        best_val_f1 = metrics["f1"]
        best_epoch = epoch

    current_lr = (
        optimizer.param_groups[0]["lr"]
    )

    epoch_seconds = (
        time.time() - epoch_start
    )

    history["epoch"].append(epoch)
    history["train_loss"].append(
        train_loss
    )
    history["train_accuracy"].append(
        train_accuracy
    )
    history["val_loss"].append(
        val_loss
    )
    history["val_accuracy"].append(
        metrics["accuracy"]
    )
    history["val_precision"].append(
        metrics["precision"]
    )
    history["val_recall"].append(
        metrics["recall"]
    )
    history["val_f1"].append(
        metrics["f1"]
    )
    history["val_balanced_accuracy"].append(
        metrics["balanced_accuracy"]
    )
    history["learning_rate"].append(
        current_lr
    )
    history["epoch_seconds"].append(
        epoch_seconds
    )

    save_checkpoint(
        epoch=epoch,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        val_loss=val_loss,
        metrics=metrics,
        is_best=is_best,
    )

    scheduler.step()

    print("=" * 80)
    print(
        f"Epoch [{epoch}/{NUM_EPOCHS}]"
    )
    print(
        f"Train Loss:            {train_loss:.4f}"
    )
    print(
        f"Train Accuracy:        {train_accuracy:.4f}"
    )
    print(
        f"Val Loss:              {val_loss:.4f}"
    )
    print(
        f"Val Accuracy:          {metrics['accuracy']:.4f}"
    )
    print(
        f"Val Catch Precision:   {metrics['precision']:.4f}"
    )
    print(
        f"Val Catch Recall:      {metrics['recall']:.4f}"
    )
    print(
        f"Val Catch F1:          {metrics['f1']:.4f}"
        + ("  <-- BEST" if is_best else "")
    )
    print(
        f"Val Balanced Accuracy: {metrics['balanced_accuracy']:.4f}"
    )
    print(
        f"Learning Rate:         {current_lr:.7f}"
    )
    print(
        f"Epoch Time:            {epoch_seconds:.1f}s"
    )

training_seconds = (
    time.time()
    - training_start
)

print("=" * 80)
print("TRAINING COMPLETE")
print(f"Best epoch:          {best_epoch}")
print(f"Best validation F1:  {best_val_f1:.4f}")
print(
    "Total training time: "
    f"{training_seconds / 60:.2f} minutes"
)
print("=" * 80)

In [ ]:
# ============================================================
# 18. RELOAD BEST CHECKPOINT
# ============================================================

best_checkpoint_path = (
    CHECKPOINTS_DIR
    / "best_checkpoint.pt"
)

if not best_checkpoint_path.exists():
    raise FileNotFoundError(
        f"Best checkpoint not found: "
        f"{best_checkpoint_path}"
    )

best_checkpoint = torch.load(
    best_checkpoint_path,
    map_location=DEVICE,
)

model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

model.eval()

print("Best checkpoint loaded.")
print(f"  Epoch:      {best_checkpoint['epoch']}")
print(
    f"  Val F1:     "
    f"{best_checkpoint['val_f1']:.4f}"
)
print(
    f"  Val Recall: "
    f"{best_checkpoint['val_recall']:.4f}"
)
print(
    f"  Val Precision: "
    f"{best_checkpoint['val_precision']:.4f}"
)

In [ ]:
# ============================================================
# 19. FINAL VALIDATION EVALUATION
# ============================================================

(
    val_loss,
    val_preds,
    val_labels,
    val_catch_probs,
) = validate_one_epoch(
    model=model,
    loader=val_loader,
    criterion=criterion,
)

final_val_metrics = calculate_metrics(
    val_labels,
    val_preds,
)

print("=" * 80)
print("BEST CHECKPOINT — VALIDATION EVALUATION")
print("=" * 80)
print(
    f"Loss:                 {val_loss:.4f}"
)
print(
    f"Accuracy:             "
    f"{final_val_metrics['accuracy']:.4f}"
)
print(
    f"Catch Precision:      "
    f"{final_val_metrics['precision']:.4f}"
)
print(
    f"Catch Recall:         "
    f"{final_val_metrics['recall']:.4f}"
)
print(
    f"Catch F1:             "
    f"{final_val_metrics['f1']:.4f}"
)
print(
    f"Balanced Accuracy:    "
    f"{final_val_metrics['balanced_accuracy']:.4f}"
)
print("=" * 80)

print("\nClassification report:")
print(
    classification_report(
        val_labels,
        val_preds,
        labels=[0, 1],
        target_names=[
            "not_catch",
            "catch",
        ],
        zero_division=0,
    )
)

In [ ]:
# ============================================================
# 20. VALIDATION CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    val_labels,
    val_preds,
    labels=[0, 1],
)

fig, ax = plt.subplots(
    figsize=(6, 5)
)

im = ax.imshow(
    cm,
    interpolation="nearest",
)

ax.set_title(
    "Validation Confusion Matrix"
)
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])

ax.set_xticklabels([
    "not_catch",
    "catch",
])

ax.set_yticklabels([
    "not_catch",
    "catch",
])

threshold = (
    cm.max() / 2.0
    if cm.max() > 0
    else 0.5
)

for row in range(2):
    for col in range(2):
        ax.text(
            col,
            row,
            str(cm[row, col]),
            ha="center",
            va="center",
            color=(
                "white"
                if cm[row, col] > threshold
                else "black"
            ),
            fontsize=13,
            fontweight="bold",
        )

fig.colorbar(im, ax=ax)
plt.tight_layout()

confusion_path = (
    REPORTS_DIR
    / "action_validation_confusion_matrix.png"
)

plt.savefig(
    confusion_path,
    dpi=180,
    bbox_inches="tight",
)

plt.show()

print(f"Saved: {confusion_path}")

In [ ]:
# ============================================================
# 21. TRAINING CURVES
# ============================================================

epochs = np.asarray(
    history["epoch"]
)

plt.figure(figsize=(8, 4))
plt.plot(
    epochs,
    history["train_loss"],
    marker="o",
    label="Train Loss",
)
plt.plot(
    epochs,
    history["val_loss"],
    marker="o",
    label="Validation Loss",
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(
    epochs,
    history["train_accuracy"],
    marker="o",
    label="Train Accuracy",
)
plt.plot(
    epochs,
    history["val_accuracy"],
    marker="o",
    label="Validation Accuracy",
)
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(
    epochs,
    history["val_f1"],
    marker="o",
    label="Validation Catch F1",
)
plt.xlabel("Epoch")
plt.ylabel("F1")
plt.title("Validation Catch F1")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(
    epochs,
    history["learning_rate"],
    marker="o",
    label="Learning Rate",
)
plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.title("Learning Rate Schedule")
plt.yscale("log")
plt.legend()
plt.tight_layout()
plt.show()

## Qualitative validation inspection

This displays a few validation clips with:
- true label
- predicted label
- confidence
- original HMDB51 action

In [ ]:
# ============================================================
# 22. QUALITATIVE VALIDATION EXAMPLES
# ============================================================

def display_clip_prediction(
    dataset,
    index,
    model,
    num_frames_to_show=4,
):
    sample = dataset[index]

    clip = sample["clip"]

    input_clip = clip.unsqueeze(0).to(
        DEVICE
    )

    with torch.no_grad():
        if amp_enabled:
            with torch.amp.autocast(
                device_type="cuda"
            ):
                logits = model(
                    input_clip
                )
        else:
            logits = model(
                input_clip
            )

        probabilities = torch.softmax(
            logits,
            dim=1,
        )[0]

    predicted_id = int(
        probabilities.argmax().item()
    )

    predicted_label = (
        IDX_TO_CLASS[predicted_id]
    )

    confidence = float(
        probabilities[
            predicted_id
        ].item()
    )

    clip_np = clip.numpy()

    clip_np = (
        clip_np.transpose(
            1, 2, 3, 0
        )
        * STD.reshape(
            1, 1, 1, 3
        )
        + MEAN.reshape(
            1, 1, 1, 3
        )
    )

    clip_np = np.clip(
        clip_np,
        0.0,
        1.0,
    )

    frame_indices = np.linspace(
        0,
        CLIP_LENGTH - 1,
        num_frames_to_show,
    ).round().astype(int)

    fig, axes = plt.subplots(
        1,
        num_frames_to_show,
        figsize=(
            4 * num_frames_to_show,
            4,
        ),
    )

    if num_frames_to_show == 1:
        axes = [axes]

    for ax, frame_index in zip(
        axes,
        frame_indices,
    ):
        ax.imshow(
            clip_np[frame_index]
        )
        ax.set_title(
            f"Frame {frame_index}"
        )
        ax.axis("off")

    fig.suptitle(
        " | ".join(
            [
                f"True: {sample['class_name']}",
                f"Pred: {predicted_label}",
                f"Conf: {confidence * 100:.1f}%",
                f"Original: {sample['original_action']}",
            ]
        ),
        fontsize=12,
    )

    plt.tight_layout()
    plt.show()

    return {
        "true_label": sample["class_name"],
        "predicted_label": predicted_label,
        "confidence": confidence,
        "original_action": sample[
            "original_action"
        ],
        "video_path": sample[
            "video_path"
        ],
    }


qualitative_count = min(
    5,
    len(val_dataset),
)

qualitative_indices = np.linspace(
    0,
    len(val_dataset) - 1,
    qualitative_count,
).round().astype(int)

qualitative_results = []

for index in qualitative_indices:
    qualitative_results.append(
        display_clip_prediction(
            val_dataset,
            int(index),
            model,
            num_frames_to_show=4,
        )
    )

## Temporal-window inference

This helper operates on exactly 16 frames and reconstructs the model from checkpoint metadata.

That makes the checkpoint usable in a fresh runtime and establishes the interface needed later for sliding-window runtime inference.

In [ ]:
# ============================================================
# 23. CHECKPOINT-BASED TEMPORAL-WINDOW INFERENCE
# ============================================================

def load_model_from_checkpoint(
    checkpoint_path,
    device=DEVICE,
):
    checkpoint_path = Path(
        checkpoint_path
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
    )

    if checkpoint.get("model_name") != "r2plus1d_18":
        raise ValueError(
            "Checkpoint does not contain R(2+1)D-18."
        )

    idx_to_class = {
        int(k): v
        for k, v in checkpoint[
            "idx_to_class"
        ].items()
    }

    num_classes = len(
        idx_to_class
    )

    model = r2plus1d_18(
        weights=None
    )

    in_features = (
        model.fc.in_features
    )

    model.fc = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(
            in_features,
            num_classes,
        ),
    )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    model.to(device)
    model.eval()

    return model, checkpoint


def preprocess_raw_frames(
    frames_rgb,
    image_size,
    mean,
    std,
):
    if len(frames_rgb) == 0:
        raise ValueError(
            "No frames supplied."
        )

    processed = []

    mean = np.asarray(
        mean,
        dtype=np.float32,
    ).reshape(1, 1, 3)

    std = np.asarray(
        std,
        dtype=np.float32,
    ).reshape(1, 1, 3)

    for frame in frames_rgb:
        frame = cv2.resize(
            frame,
            (
                image_size,
                image_size,
            ),
            interpolation=cv2.INTER_LINEAR,
        )

        frame = (
            frame.astype(np.float32)
            / 255.0
        )

        frame = (
            frame - mean
        ) / std

        processed.append(frame)

    clip = np.stack(
        processed,
        axis=0,
    )

    # [T,H,W,C] -> [1,C,T,H,W]
    clip = np.transpose(
        clip,
        (3, 0, 1, 2),
    )

    return torch.from_numpy(
        clip.copy()
    ).float().unsqueeze(0)


def predict_temporal_window(
    frames_rgb,
    checkpoint_path,
):
    model, checkpoint = (
        load_model_from_checkpoint(
            checkpoint_path,
            device=DEVICE,
        )
    )

    clip_length = int(
        checkpoint["clip_length"]
    )

    if len(frames_rgb) != clip_length:
        raise ValueError(
            f"Expected {clip_length} frames, "
            f"got {len(frames_rgb)}."
        )

    clip = preprocess_raw_frames(
        frames_rgb=frames_rgb,
        image_size=int(
            checkpoint["image_size"]
        ),
        mean=checkpoint["mean"],
        std=checkpoint["std"],
    ).to(DEVICE)

    with torch.no_grad():
        if (
            USE_AMP
            and DEVICE.type == "cuda"
        ):
            with torch.amp.autocast(
                device_type="cuda"
            ):
                logits = model(clip)
        else:
            logits = model(clip)

        probabilities = torch.softmax(
            logits,
            dim=1,
        )[0]

    predicted_id = int(
        probabilities.argmax().item()
    )

    idx_to_class = {
        int(k): v
        for k, v in checkpoint[
            "idx_to_class"
        ].items()
    }

    return {
        "class_id": predicted_id,
        "label": idx_to_class[
            predicted_id
        ],
        "confidence": float(
            probabilities[
                predicted_id
            ].item()
        ),
        "probabilities": {
            idx_to_class[i]: float(
                probabilities[i].item()
            )
            for i in range(
                len(probabilities)
            )
        },
    }

print(
    "Checkpoint-based temporal-window inference is ready."
)

In [ ]:
# ============================================================
# 24. OPTIONAL: TEMPORAL-WINDOW TEST
# ============================================================

RUN_EXAMPLE_TEMPORAL_INFERENCE = True

if RUN_EXAMPLE_TEMPORAL_INFERENCE:
    example_video_path = Path(
        val_df.iloc[0]["video_path"]
    )

    cap = cv2.VideoCapture(
        str(example_video_path)
    )

    if not cap.isOpened():
        raise RuntimeError(
            f"Could not open example video: "
            f"{example_video_path}"
        )

    frames_rgb = []

    for _ in range(CLIP_LENGTH):
        ok, frame = cap.read()

        if not ok:
            break

        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB,
        )

        frames_rgb.append(frame)

    cap.release()

    if len(frames_rgb) == CLIP_LENGTH:
        result = predict_temporal_window(
            frames_rgb,
            best_checkpoint_path,
        )

        print(
            json.dumps(
                result,
                indent=2,
            )
        )
    else:
        print(
            "Example video did not provide "
            "a complete 16-frame window; "
            "inference example skipped."
        )

In [ ]:
# ============================================================
# 25. SAVE REPORTS
# ============================================================

summary = {
    "task":
        "binary_temporal_action_recognition",

    "positive_class":
        "catch",

    "negative_class":
        "not_catch",

    "negative_source_actions":
        NEGATIVE_ACTIONS,

    "model":
        MODEL_NAME,

    "pretrained":
        PRETRAINED,

    "freeze_backbone":
        FREEZE_BACKBONE,

    "clip_length":
        CLIP_LENGTH,

    "image_size":
        IMAGE_SIZE,

    "normalization": {
        "mean": MEAN.tolist(),
        "std": STD.tolist(),
    },

    "random_seed":
        RANDOM_SEED,

    "split_counts": {
        "train": len(train_df),
        "val": len(val_df),
        "test": len(test_df),
    },

    "train_class_distribution": {
        str(k): int(v)
        for k, v in (
            train_df["class_id"]
            .value_counts()
            .sort_index()
            .items()
        )
    },

    "val_class_distribution": {
        str(k): int(v)
        for k, v in (
            val_df["class_id"]
            .value_counts()
            .sort_index()
            .items()
        )
    },

    "test_class_distribution": {
        str(k): int(v)
        for k, v in (
            test_df["class_id"]
            .value_counts()
            .sort_index()
            .items()
        )
    },

    "best_epoch":
        int(best_epoch),

    "best_validation_metrics": {
        "accuracy": float(
            best_checkpoint[
                "val_accuracy"
            ]
        ),
        "precision": float(
            best_checkpoint[
                "val_precision"
            ]
        ),
        "recall": float(
            best_checkpoint[
                "val_recall"
            ]
        ),
        "f1": float(
            best_checkpoint[
                "val_f1"
            ]
        ),
        "balanced_accuracy": float(
            best_checkpoint[
                "val_balanced_accuracy"
            ]
        ),
    },

    "training_duration_seconds":
        float(training_seconds),

    "checkpoint":
        str(best_checkpoint_path),
}

summary_path = (
    REPORTS_DIR
    / "action_training_summary.json"
)

history_path = (
    REPORTS_DIR
    / "action_training_history.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        summary,
        f,
        indent=2,
    )

with open(
    history_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        history,
        f,
        indent=2,
    )

print(f"Saved summary: {summary_path}")
print(f"Saved history: {history_path}")

# Notebook 05 completion status

Expected artifacts:

```text
training/checkpoints/action_model/
    last_checkpoint.pt
    best_checkpoint.pt

training/reports/
    action_training_summary.json
    action_training_history.json
    action_validation_confusion_matrix.png
```

Notebook 05 ends at temporal action recognition.

Notebook 06 owns final held-out test evaluation.